## Training Speech AI with Mozilla Data Collective

This tutorial walks you through downloading, loading and finetuning Whisper on an MDC dataset end to end. In this notebook we will use as an example use-case the [Khmer](https://en.wikipedia.org/wiki/Khmer_language) language (an official and national language in Cambodia)  using the [Khmer ASR Cultural Dataset](https://datacollective.mozillafoundation.org/datasets/cmkcy8in2004umo0775mye43g) steward by [Digital Divide Data](https://www.digitaldividedata.com/)

Specifically, the flow of this tutorial focuses on:

1. Ensuring a GPU set up is available
2. Setting up and logging in to Mozilla Data Collective
3. Downloading and loading the dataset as a pandas DataFrame - with a single function call!
4. Generating an automated Exploratory Data Analysis (EDA) report to understand the dataset and inform our finetuning configuration
5. Configuring Whisper's fine-tuning hyper-parameters
6. Launching a fine-tuning job!

Note that steps 1-4 are only required the first time you set up your environment and download the dataset. Once you have the dataset downloaded and ready to use, you can skip directly to steps 5: to create a new set of hyper-parameters and 6: to start a new finetuning job!


### 1. Confirm GPU availability

If you are running this notebook in Google Collab you'll need to enable GPUs for the notebook: Navigate to Edit→Notebook Settings Select T4 GPU from the Hardware Accelerator section Click Save and accept. Next, we'll confirm that we can connect to the GPU:

In [ ]:
import torch

if not torch.cuda.is_available():
    print("GPU NOT available!")
else:
    print("GPU is available!")

### 2. Setup and login to Mozilla Data Collective

***(Required)*** In order to download any MDC dataset you will need to first create an account at Mozilla Data Collective and then get an API key.

1. Create a Mozilla Data Collective [account](https://datacollective.mozillafoundation.org/)
2. Get your API key by following the instructions [here](https://datacollective.mozillafoundation.org/api-reference)
3. Set your API key as an environment variable in your .env file or export it in your terminal. If you are running this notebook in Google Collab, you can set it for the session by running the cell below and entering your API key when prompted.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # this will load environment variables from your .env file if you have one

if not os.getenv(
    "MDC_API_KEY"
):  # if API key is not set you can set it through the interactive jupyter terminal
    from getpass import getpass

    os.environ["MDC_API_KEY"] = getpass("Enter your Mozilla Data Collective API key: ")

***(Optional)*** If you want to track training and evaluation metrics of the finetuning and save your final model to use it and share it with others later, you will need a Hugging Face (HF) account.
1. Create a HF [account](https://huggingface.co/join)
2. Set up [personal access token](huggingface.co/settings/tokens)
3. Login to hugging face in this notebook by running the command below and using your token


In [ ]:
!huggingface-cli login

### 3. Download and load the Khmer dataset as a DataFrame

In [ ]:
from datacollective import load_dataset

dataframe = load_dataset(
    "khmer-asr-cultural-dataset-4e33cd05",
    download_directory="local_data",
    enable_logging=True,
)

print("MDC dataframe preview:", dataframe.head())

### 4. Exploratory Data Analysis (EDA) with ydata-profiling

In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(dataframe, title="Profiling Report")
profile.to_file("khmer_dataset_report.html")

### 5. Configure hyper-parameters for finetuning

In [4]:
# @title Finetuning configuration and hyperparameter setting
import yaml


def save_to_yaml(filename="config.yaml"):
    with open(filename, "w") as file:
        yaml.dump(cfg, file)


model_id = "openai/whisper-tiny"  # @param ["openai/whisper-tiny", "openai/whisper-small", "openai/whisper-medium","openai/whisper-large-v3"]
dataset_id = "khmer-asr-cultural-dataset-4e33cd05"  # @param {type: "string"}
language = "Khmer"  # @param {type: "string"}
repo_name = "default"  # @param {type: "string"}
push_to_hub = False  # @param {type: 'boolean'}
n_train_samples = 50  # @param {type: "int"}
n_test_samples = 10  # @param {type: "int"}
download_directory = "local_data"  # @param {type: "string"}
hub_private_repo = True  # @param {type: 'boolean'}
max_steps = 25  # @param {type: "slider", min: 1, max: 3000, step: 10}
per_device_train_batch_size = 16  # @param {type: "slider", min: 1, max: 300}
gradient_accumulation_steps = 1  # @param {type: "slider", min: 1, max: 10}
warmup_steps = 5  # @param {type: "slider", min: 0, max: 500}
gradient_checkpointing = True  # @param {type: 'boolean'}
fp16 = True  # @param {type: 'boolean'}
per_device_eval_batch_size = 8  # @param {type: "slider", min: 1, max: 200}
save_steps = 5  # @param {type: "slider", min: 1, max: 500}
logging_steps = 5  # @param {type: "slider", min: 1, max: 500}
load_best_model_at_end = True  # @param {type: 'boolean'}

cfg = {
    "model_id": model_id,
    "dataset_id": dataset_id,
    "language": language,
    "repo_name": repo_name,
    "n_train_samples": n_train_samples,
    "n_test_samples": n_test_samples,
    "download_directory": download_directory,
    "training_hp": {
        "push_to_hub": push_to_hub,
        "hub_private_repo": hub_private_repo,
        "max_steps": max_steps,
        "per_device_train_batch_size": per_device_train_batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "learning_rate": 1e-5,
        "warmup_steps": warmup_steps,
        "gradient_checkpointing": gradient_checkpointing,
        "fp16": fp16,
        "eval_strategy": "steps",
        "per_device_eval_batch_size": per_device_eval_batch_size,
        "predict_with_generate": True,
        "generation_max_length": 225,
        "save_steps": save_steps,
        "logging_steps": logging_steps,
        "load_best_model_at_end": load_best_model_at_end,
        "save_total_limit": 1,
        "metric_for_best_model": "wer",
        "greater_is_better": False,
    },
}

save_to_yaml()

### 6. Start finetuning job

Note that this might take a while, anything from 10min to 10hours depending on your model choice and hyper-parameter configuration

In [3]:
from speech_to_text_finetune.finetune_whisper import run_finetuning

run_finetuning(config_path="config.yaml")

2026-04-06 19:47:48.612 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:65 - Finetuning starts soon, results saved locally at ./artifacts/whisper-tiny-km
2026-04-06 19:47:48.614 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:76 - Loading openai/whisper-tiny on NVIDIA GeForce RTX 2060 SUPER.
2026-04-06 19:47:49.429 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:82 - Configuring openai/whisper-tiny for Khmer
2026-04-06 19:47:51.669 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:120 - Loading khmer-asr-cultural-dataset-4e33cd05.
2026-04-06 19:47:56.086 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:128 - Processing dataset...
Saving the dataset (1/1 shards): 100%|██████████| 20/20 [00:00<00:00, 400.74 examples/s]
2026-04-06 19:48:04.250 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:135 - Processed dataset saved at artifacts/khmer-asr-cultural-dataset-4e33cd05

2026-04-06 19:48:12.378 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:166 - Baseline evaluation complete. Results:
	 {'eval_loss': 3.0822842121124268, 'eval_model_preparation_time': 0.0026, 'eval_wer_ortho': 590.4761904761905, 'eval_wer': 133.96481732070365, 'eval_cer_ortho': 166.04970914859862, 'eval_cer': 154.1460735859418, 'eval_runtime': 5.4628, 'eval_samples_per_second': 3.661, 'eval_steps_per_second': 0.549}
2026-04-06 19:48:12.379 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:168 - Start finetuning job on 50 audio samples. Monitor training metrics in real time in a local tensorboard server by running in a new terminal: tensorboard --logdir ./artifacts/whisper-tiny-km/runs


Step,Training Loss,Validation Loss,Model Preparation Time,Wer Ortho,Wer,Cer Ortho,Cer
5,3.063300,3.082284,0.002600,590.476190,133.964817,166.049709,154.146074
10,2.607700,2.371143,0.002600,375.000000,115.832206,122.845056,107.193850
15,2.213100,2.175217,0.002600,585.714286,120.703654,180.803808,169.467326
20,2.082600,2.078974,0.002600,504.761905,120.027064,173.717610,155.848435
25,2.000900,2.011298,0.002600,367.857143,100.000000,126.599683,115.156507
30,1.929100,1.961748,0.002600,819.047619,142.354533,206.716023,188.467875
35,1.895000,1.931766,0.002600,634.523810,125.439783,146.060286,135.365184
40,1.852400,1.913710,0.002600,635.714286,125.439783,145.742993,134.980780
45,1.842600,1.903880,0.002600,626.190476,125.439783,144.896880,134.541461
50,1.821900,1.899398,0.002600,617.857143,125.439783,144.526705,134.321801


/home/kostis/Projects/MDC/mdc-stt-finetune/.venv/lib/python3.13/site-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50358, 50359, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['proj_out.w

2026-04-06 19:50:51.543 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:181 - Evaluation complete. Results:
	 {'eval_loss': 2.011298418045044, 'eval_model_preparation_time': 0.0026, 'eval_wer_ortho': 367.85714285714283, 'eval_wer': 100.0, 'eval_cer_ortho': 126.59968270756214, 'eval_cer': 115.15650741350908, 'eval_runtime': 4.5502, 'eval_samples_per_second': 4.395, 'eval_steps_per_second': 0.659, 'epoch': 12.5}
2026-04-06 19:50:51.546 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:202 - Find your final, best performing model at ./artifacts/whisper-tiny-km


({'eval_loss': 3.0822842121124268,
  'eval_model_preparation_time': 0.0026,
  'eval_wer_ortho': 590.4761904761905,
  'eval_wer': 133.96481732070365,
  'eval_cer_ortho': 166.04970914859862,
  'eval_cer': 154.1460735859418,
  'eval_runtime': 5.4628,
  'eval_samples_per_second': 3.661,
  'eval_steps_per_second': 0.549},
 {'eval_loss': 2.011298418045044,
  'eval_model_preparation_time': 0.0026,
  'eval_wer_ortho': 367.85714285714283,
  'eval_wer': 100.0,
  'eval_cer_ortho': 126.59968270756214,
  'eval_cer': 115.15650741350908,
  'eval_runtime': 4.5502,
  'eval_samples_per_second': 4.395,
  'eval_steps_per_second': 0.659,
  'epoch': 12.5})